# Day 1, morning: see what your code does

This notebook is a **workbench**: you write no code here. Your code goes in the package, and each section
below calls it and draws what comes out. The figures follow lecture L1.

| You implement | In | L1 slides | Section here |
|---|---|---|---|
| `pareto_mask_array` | `common/pareto.py` | Pareto fronts | 6 |
| `recognise_isolated` | `speech/gmm_hmm.py` | isolated words | 5 |
| `NGramLM.fit`, `prob`, `perplexity` | `speech/ngram.py` | n-grams, perplexity | 7 |

**Reference solution (since 17 November).** It is on branch `solutions` of the starter repository, not on
`main`, so pulling `main` never replaces your code. To run this notebook on it, or to compare with your own:
`git fetch course`, then `git checkout course/solutions -- <file>` for the file you want, or
`git diff HEAD course/solutions -- <file>` to compare. With your own code, every cell that reaches a function
you have not implemented says so and the notebook goes on.

**What "implement" means.** Each function's body is one `raise NotImplementedError("TODO(d1): ...")`, and its
message, the docstring and the L1 slide named in the table say what to write. Replace the `raise` by your code,
then re-run the cell. (The "holes" in the plan below are the steps of each function, in the order to write them.)

**Plan for the lab (2 h).**

| Time | What | Computer time |
|---|---|---|
| 0:00–0:15 | Sections 1–4 (provided code): run, look | 1 min |
| 0:15–0:25 | Section 5: `recognise_isolated` (1 hole). Then start the sweep of section 6 in a terminal | sweep about 5 min |
| 0:25–0:40 | Section 6: `pareto_mask_array` (1 hole), the front and its intervals | |
| 0:40–1:15 | Section 7: `NGramLM` (4 holes: `fit` 1, `prob` 2, `perplexity` 1), then `--part lm` | `--part lm` a few s |
| 1:15–1:40 | `pytest -m d1 -k "pareto or ngram or isolated"`; the questions of sections 4–7 | |
| 1:40–2:00 | One `DECISIONS.md` entry; commit | |

Sections 1–4 use only provided code: run them first, while you read the lecture.
A cell that reaches one of your unfinished functions says which one, and the rest of the notebook keeps
working. After editing the package, simply re-run the cell: `autoreload` picks up your changes.

Data: `python scripts/fetch_data.py data` (or set `DSP_FSDD` to a folder of FSDD wav files).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "speech").is_dir() else Path.cwd().parent  # the folder that contains speech/
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # results/ and data/ are relative to the bundle root

import librosa
from IPython.display import Audio, display

from asr_demo import digits  # provided: FSDD paths, training data and cached word models (shared with the demo)
from speech import run_classical_baseline as R
from speech.gmm_hmm import left_to_right_hmm, mfcc_features, train_word_models, uniform_segmentation_init

SR = R.SR
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm, imported by a dependency


def not_yet(e):
    """What a cell prints when it reaches one of your stubs."""
    print(f"Not implemented yet: {e}\nImplement it in the package, then re-run this cell.")


print("FSDD recordings:", digits.FSDD, "| found:", len(list(digits.FSDD.glob("*.wav"))))

## 1. Listen and look

The Free Spoken Digit Dataset: 6 speakers, 50 recordings of each digit per speaker, 8 kHz.
Files `<digit>_<speaker>_<index>.wav`; indices 0–4 are the official test split.
The loader trims leading and trailing silence and scales every word to the same loudness.

In [ ]:
data = R.load_fsdd(digits.FSDD)
train_set = [d for d in data if d["idx"] >= 5]
test_set = [d for d in data if d["idx"] < 5]
print(len(train_set), "training recordings,", len(test_set), "test recordings")

sevens = [d for d in train_set if d["digit"] == "7" and d["idx"] == 5][:3]
fig, axes = plt.subplots(1, 3, figsize=(12, 2.4), sharey=True)
for ax, d in zip(axes, sevens):
    t = np.arange(len(d["y"])) / SR
    ax.plot(t, d["y"], lw=0.6)
    ax.set_title(f'"seven", speaker {d["speaker"]}')
    ax.set_xlabel("time (s)")
fig.tight_layout()
for d in sevens:
    display(Audio(d["y"], rate=SR))

## 2. MFCC, step by step

The provided `mfcc_features` calls `librosa`. Here we redo its steps one by one (L1, "MFCC step by step"),
then check that we get the same numbers. Window 25 ms (200 samples), hop 10 ms (80 samples), Hann window,
40 mel bands, 13 coefficients.

In [ ]:
y = sevens[0]["y"]
n_fft, hop = 200, 80
P = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, win_length=n_fft, window="hann")) ** 2  # power spectrum
H = librosa.filters.mel(sr=SR, n_fft=n_fft, n_mels=40, fmax=SR / 2)  # 40 triangular filters
E = H @ P                                                             # mel energies
logE = librosa.power_to_db(E, top_db=80.0)                            # log (dB), 80 dB floor
C = librosa.feature.mfcc(S=logE, n_mfcc=13)                           # DCT-II, keep 13

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
axes[0].imshow(librosa.power_to_db(P), origin="lower", aspect="auto", cmap="magma")
axes[0].set(title="power spectrum $P_t[f]$ (dB)", xlabel="frame $t$", ylabel="FFT bin $f$")
axes[1].plot(H.T, lw=0.8)
axes[1].set(title="mel filters $H_\\nu[f]$", xlabel="FFT bin $f$")
axes[2].imshow(logE, origin="lower", aspect="auto", cmap="magma")
axes[2].set(title="log-mel $\\tilde E_t[\\nu]$", xlabel="frame $t$", ylabel="mel band $\\nu$")
axes[3].imshow(C[1:], origin="lower", aspect="auto", cmap="coolwarm", extent=(0, C.shape[1], 0.5, 12.5))
axes[3].set(title="MFCC $c_t[\\ell]$, $\\ell=1..12$ ($c_0$: energy, not shown)", xlabel="frame $t$", ylabel="coefficient $\\ell$")
fig.tight_layout()

same = np.allclose(C.T, mfcc_features(y, SR, deltas=False))
print("Same as the provided mfcc_features:", same)

**Question.** The DCT keeps 13 of 40 coefficients. Invert it (`librosa.feature.inverse.mfcc_to_mel`) and compare
with the log-mel image: what is lost? Why is that useful for recognising *words* rather than *speakers*?

With deltas, every frame becomes a 39-dimensional vector $\mathbf{x}_t$. The models below use them after a
**global** normalisation (mean and variance estimated on all training frames).

In [ ]:
tr_files, train_feats, norm = digits.training_data()  # provided: normalised features per word, + silence
X7 = train_feats["7"]
print(f'"seven": {len(X7)} training recordings, {sum(len(x) for x in X7)} frames of dimension {X7[0].shape[1]}')

## 3. EM on a Gaussian mixture

One state of a word HMM is a GMM over frames (L1, "EM on one GMM"). To *see* EM, take every frame of
"seven" in two dimensions only ($c_1$, $c_2$), fit $M=3$ Gaussians, and record the log-likelihood after
every iteration. `GaussianMixture(max_iter=1, warm_start=True)` performs exactly one E-step and one M-step per
call. Three random initialisations show the **local optima**.

In [ ]:
from matplotlib.patches import Ellipse
from sklearn.mixture import GaussianMixture

frames = np.concatenate(X7)[:, 1:3]


def em_trace(seed, n_iter=40, M=3):
    g = GaussianMixture(M, covariance_type="diag", max_iter=1, warm_start=True, random_state=seed,
                        init_params="random_from_data")
    ll, snaps = [], {}
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # "did not converge": expected, we stop after every iteration
        for it in range(1, n_iter + 1):
            g.fit(frames)
            ll.append(g.score(frames) * len(frames))
            if it in (1, 5, n_iter):
                snaps[it] = (g.means_.copy(), g.covariances_.copy())
    return ll, snaps


traces = {s: em_trace(s) for s in (0, 1, 2)}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for s, (ll, _) in traces.items():
    axes[0].plot(range(1, len(ll) + 1), ll, label=f"init {s}")
axes[0].set(title="training log-likelihood", xlabel="EM iteration")
axes[0].legend()
_, snaps = traces[0]
for ax, it in zip(axes[1:], (1, 5, 40)):
    ax.scatter(frames[::4, 0], frames[::4, 1], s=2, alpha=0.3, color="gray")
    for mu, var in zip(*snaps[it]):
        ax.add_patch(Ellipse(mu, 4 * np.sqrt(var[0]), 4 * np.sqrt(var[1]), fill=False, lw=2))
    ax.set(title=f"init 0, after {it} iteration(s)", xlabel="$c_1$", ylabel="$c_2$")
fig.tight_layout()
print("final log-likelihood per initialisation:", {s: round(ll[-1]) for s, (ll, _) in traces.items()})

**What to see.** The curves never go down (EM's guarantee). They do not all end at the same value: EM finds a
*local* optimum. In the word models, the provided **flat start** chooses the initialisation for you (next section).

## 4. A word HMM, and how it cuts a word into states

Train one 5-state left-to-right GMM-HMM for "seven" with the provided code (flat start, then Baum–Welch),
and look at:
- the transition matrix $A$: only "stay" ($a_{jj}$) and "move" ($a_{j,j+1}$) are non-zero, and the expected time
  in state $j$ is $1/(1-a_{jj})$ frames;
- the best state path (Viterbi, `model.decode`) on a test recording, before and after training.

In [ ]:
wm7 = train_word_models({"7": X7}, n_states=5, n_mix=2)
hmm7 = wm7.models["7"]
flat = left_to_right_hmm(5, 2)        # initial A: stay 0.6, move 0.4
uniform_segmentation_init(flat, X7)   # flat start: one GMM per equal chunk, before any Baum-Welch iteration

test7 = next(d for d in test_set if d["digit"] == "7")
x = norm(mfcc_features(test7["y"], SR))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2), gridspec_kw={"width_ratios": [1, 2, 2]})
axes[0].imshow(hmm7.transmat_, cmap="Blues", vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        if hmm7.transmat_[i, j] > 0.005:
            axes[0].text(j, i, f"{hmm7.transmat_[i, j]:.2f}", ha="center", va="center", fontsize=8)
axes[0].set(title="trained $A$", xlabel="to state", ylabel="from state")
for ax, (name, m) in zip(axes[1:], (("flat start", flat), ("after Baum-Welch", hmm7))):
    logp, path = m.decode(x)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=test7["y"], sr=SR, n_fft=200, hop_length=80, n_mels=40))
    ax.imshow(mel[:, :len(path)], origin="lower", aspect="auto", cmap="magma")
    for b in np.flatnonzero(np.diff(path)) + 0.5:
        ax.axvline(b, color="w", lw=2)
    for j in range(5):
        ax.text(np.mean(np.flatnonzero(path == j)), 36, str(j), color="w", ha="center", fontsize=11, weight="bold")
    ax.set(title=f"{name}: best state path, log p = {logp:.0f}", xlabel="frame $t$", ylabel="mel band")
fig.tight_layout()
print("expected frames per state:", np.round(1 / (1 - np.diag(hmm7.transmat_)[:-1]), 1))

**Question.** Where do the boundaries move during training, and why does the log-likelihood increase?
Which state lasts longest, and which sound of "seven" could it be?

## 5. Your recogniser: `recognise_isolated`

Ten word models (5 states, 2 Gaussians each; trained once and cached in `results/speech_classical/`, about
20 s the first time). For every test recording your function must return the digit whose model gives the
highest $\log p(X\mid\theta_w)$.

In [ ]:
from speech.gmm_hmm import recognise_isolated

wm, norm = digits.word_models(5, 2)
test_feats = [norm(mfcc_features(d["y"], SR)) for d in test_set]
truth = [d["digit"] for d in test_set]
try:
    pred = [recognise_isolated(wm, f, R.DIGITS) for f in test_feats]
except NotImplementedError as e:
    pred = None
    not_yet(e)

if pred is not None:
    conf = np.zeros((10, 10), int)
    for t, p in zip(truth, pred):
        conf[int(t), int(p)] += 1
    acc = np.trace(conf) / conf.sum()
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(conf, cmap="Blues")
    for i in range(10):
        for j in range(10):
            if conf[i, j]:
                ax.text(j, i, conf[i, j], ha="center", va="center", fontsize=7,
                        color="white" if conf[i, j] > 15 else "black")
    ax.set(title=f"accuracy {acc:.3f} on {len(truth)} test files", xlabel="recognised", ylabel="said",
           xticks=range(10), yticks=range(10))
    fig.tight_layout()

**What your function compared**, for one test recording: the ten log-likelihoods. How far is the winner from
the second best? Find a recording that your recogniser gets wrong and look at its bars.

In [ ]:
i = 0  # change me
scores = {w: wm.models[w].score(test_feats[i]) for w in R.DIGITS}
best = max(scores.values())
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.bar(list(scores), [v - best for v in scores.values()], color=["C3" if w == truth[i] else "C0" for w in scores])
ax.plot(truth[i], scores[truth[i]] - best, "v", color="C3", ms=9)  # visible even when it is the winner (height 0)
ax.set(title=f'test file {i}: said "{truth[i]}" (red)', xlabel="word model $w$",
       ylabel="$\\log p(X\\mid\\theta_w)$ minus the best")
fig.tight_layout()

## 6. Your Pareto front: `pareto_mask_array`

Run the sweep first (it writes `results/speech_classical/isolated_digits.csv` and the rows of `results.jsonl`):
```
export DSP_MILESTONE=d1
python -m speech.run_classical_baseline --fsdd data/fsdd/recordings --part isolated --states 2 3 5 8 --mixes 1 2 4
```
Then this cell keeps the configurations that no other one beats on both latency (lower is better) and
accuracy (higher is better), with your `pareto_mask_array`, and draws the 95 % Wilson interval of each accuracy.

In [ ]:
import pandas as pd

from common.pareto import pareto_mask_array

csv = ROOT / "results/speech_classical/isolated_digits.csv"
if not csv.exists():
    print(f"No {csv.relative_to(ROOT)} yet: run the sweep above, then re-run this cell.")
else:
    df = pd.read_csv(csv)
    df = df[(df.split == "standard") & (df.snr.astype(str) == "clean")].drop_duplicates("name", keep="last")
    try:
        front = pareto_mask_array(df[["lat_ms", "acc"]].to_numpy(), ["min", "max"])
    except NotImplementedError as e:
        front = None
        not_yet(e)
    n, z = len(test_set), 1.96
    p = df.acc.to_numpy()
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(df.lat_ms, p, yerr=[p - (centre - half), centre + half - p], fmt="o", color="gray", capsize=2)
    if front is not None:
        f = df[front].sort_values("lat_ms")
        ax.step(f.lat_ms, f.acc, where="post", color="k", ls="--", label="your front")
        ax.plot(f.lat_ms, f.acc, "o", color="C0", zorder=3)
        ax.legend()
    for _, r in df.iterrows():
        ax.annotate(r["name"], (r.lat_ms, r.acc), fontsize=7, xytext=(3, -8), textcoords="offset points")
    ax.set(xlabel="latency per word (ms), lower is better", ylabel="accuracy, higher is better")
    fig.tight_layout()

**Question.** One error out of 300 files is 0.33 point. Which points of your front are really different from
each other? Which configuration would you choose under a latency budget, and how sure are you?

## 7. Your n-gram models: `NGramLM`

A digit "grammar" in which the next digit depends on the previous **two** (provided `second_order_chain`).
Your `NGramLM` learns it from samples; perplexity is measured on 1000 other sentences
(L1, "Perplexity versus order, data size and smoothing").

In [ ]:
from speech.ngram import NGramLM, sample_second_order, second_order_chain

rng = np.random.default_rng(0)
grammar = second_order_chain(rng)
test_text = sample_second_order(rng, grammar, 1000)
print("three sentences of the grammar:", [" ".join(s) for s in test_text[:3]])

results = {}
try:
    for n_train in (50, 500, 5000):
        train_text = sample_second_order(rng, grammar, n_train)
        for smoothing in ("add-k", "interpolated"):
            results[n_train, smoothing] = [NGramLM(order, smoothing).fit(train_text, R.DIGITS).perplexity(test_text)
                                           for order in range(1, 6)]
except NotImplementedError as e:
    not_yet(e)

if results:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
    for ax, smoothing in zip(axes, ("add-k", "interpolated")):
        for n_train in (50, 500, 5000):
            ax.plot(range(1, 6), results[n_train, smoothing], "o-", label=f"{n_train} training sentences")
        ax.set(title=smoothing, xlabel="order $n$", ylabel="test perplexity", xticks=range(1, 6))
        ax.legend()
    fig.tight_layout()

**Questions.**
- Which order wins with 5000 sentences, and why can no model go much below the grammar's own perplexity?
- Why does the 5-gram get *worse* with 50 sentences, and why does interpolation hurt it less than add-$k$?

## Before you leave

Write at least one entry in `DECISIONS.md` (format in `SUBMISSION_SPEC.md`), e.g. which ($S$, $M$) you would
deploy under a latency budget, with the rows or the figure that justify it.